In [1]:
import torch
from datasets import load_dataset
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    Trainer,
    TrainingArguments
)

# 1. Load the IMDb dataset
dataset = load_dataset("imdb")

# The dataset["train"] subset has 25,000 samples. We'll split it into two halves.
full_train_data = dataset["train"]

/Users/dariofenoglio/miniforge3/envs/eris/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Shuffle the training set for a fair split
full_train_data = full_train_data.shuffle(seed=42)

# Split into IN set (first half) and OUT set (second half)
half_size = len(full_train_data) // 2
in_set = full_train_data.select(range(0, half_size))
out_set = full_train_data.select(range(half_size, len(full_train_data)))

# We will train on in_set, and out_set will be used only for membership inference evaluation
test_data = dataset["test"]  # Standard test set (25,000 samples) for normal model evaluation if desired

# 2. Initialize tokenizer and model
model_name = "distilbert-base-uncased"
tokenizer = DistilBertTokenizerFast.from_pretrained(model_name)
model = DistilBertForSequenceClassification.from_pretrained(model_name, num_labels=2)

# 3. Preprocessing (tokenization) function
def tokenize_function(example):
    return tokenizer(
        example["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

# Tokenize IN and OUT sets
in_set = in_set.map(tokenize_function, batched=True)
out_set = out_set.map(tokenize_function, batched=True)

# (Optional) Tokenize standard test set
test_data = test_data.map(tokenize_function, batched=True)

# Set formats for PyTorch
in_set.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
out_set.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
test_data.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Map: 100%|██████████| 12500/12500 [00:01<00:00, 6433.58 examples/s]


In [3]:
# 4. Define training arguments
training_args = TrainingArguments(
    output_dir="./distilbert-imdb",
    overwrite_output_dir=True,
    num_train_epochs=2,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_dir="./logs",
    logging_steps=100,
    seed=42
)

# 5. Trainer initialization using only the IN set for training
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=in_set,
    eval_dataset=test_data,  # Normal evaluation on the official test set
)

# 6. Train the model
trainer.train()



/Users/dariofenoglio/miniforge3/envs/eris/lib/python3.13/site-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


Epoch,Training Loss,Validation Loss
1,0.465600,0.502011
2,0.319800,0.598972


TrainOutput(global_step=6250, training_loss=0.4083143977355957, metrics={'train_runtime': 674.0469, 'train_samples_per_second': 37.089, 'train_steps_per_second': 9.272, 'total_flos': 827921241600000.0, 'train_loss': 0.4083143977355957, 'epoch': 2.0})

In [4]:
# ---------------------------------------------------------------
# Membership Inference Attack (MIA) - Loss-based
# ---------------------------------------------------------------

from tqdm import tqdm
import numpy as np
from torch.utils.data import DataLoader

# Helper function: compute average loss on a dataset
def compute_loss(trainer, dataset):
    # We'll create our own DataLoader for dataset
    dataloader = DataLoader(dataset, batch_size=16)
    losses = []
    model.eval()
    with torch.no_grad():
        for batch in tqdm(dataloader):
            input_ids = batch["input_ids"].to(trainer.args.device)
            attention_mask = batch["attention_mask"].to(trainer.args.device)
            labels = batch["label"].to(trainer.args.device)
            outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
            losses.append(loss.item())
    return np.mean(losses)

# Move model to the same device as the trainer
model.to(trainer.args.device)

# Compute the average loss for the in_set
in_loss = compute_loss(trainer, in_set)
# Compute the average loss for the out_set
out_loss = compute_loss(trainer, out_set)

print(f"Average Loss (IN set):  {in_loss:.4f}")
print(f"Average Loss (OUT set): {out_loss:.4f}")

# Typically, the model's loss on data it has seen (IN set) is lower than on data it has NOT seen (OUT set).
# You can threshold the loss difference to decide membership in practice.

Average Loss (IN set):  0.1344
Average Loss (OUT set): 0.5931


In [5]:
def get_parameters(model, config):
    return [val.cpu().numpy() for _, val in model.state_dict().items()]

p = get_parameters(model, None)



In [16]:
trainer.args.device

device(type='mps')

In [7]:
from torch.utils.data import Dataset
from transformers import DistilBertTokenizerFast
import torch

class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer=None, max_length=128):
        self.texts = texts
        labels = labels
        tokenizer = tokenizer or DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]

        # Tokenize on-the-fly
        encoding = self.tokenizer(
            text,
            padding='max_length',
            truncation=True,
            max_length=self.max_length,
            return_tensors='pt'
        )

        # Squeeze out the batch dimension returned by the tokenizer
        input_ids = encoding['input_ids'].squeeze(0)
        attention_mask = encoding['attention_mask'].squeeze(0)

        # Return ((input_ids, attention_mask), label)
        return (input_ids, attention_mask), label

In [14]:
from torch.utils.data import random_split, ConcatDataset, DataLoader
import numpy as np

# Example arrays of text & labels
all_texts = test_data['text'] 
all_labels = test_data['label']

# Create the full dataset
full_dataset = TextDataset(all_texts, all_labels)

# Suppose self.canary_frac is some integer count or fraction
canary_count = 100  # Just an example
non_canary_count = len(full_dataset) - canary_count

canaries, non_canaries = random_split(
    full_dataset, 
    [canary_count, non_canary_count]
)

# Keep track of # canaries
n_canaries = len(canaries)
scores = np.zeros(n_canaries)

# Bernoulli(0.5) => half are truly "in" (1) and half "out" (0)
true_in_out = torch.distributions.bernoulli.Bernoulli(
    torch.ones(n_canaries) * 0.5
).sample()

true_in_out = true_in_out.numpy()  # store for MIA

canaries_in_idx = torch.nonzero(torch.tensor(true_in_out)).squeeze(-1)  # Indices of "in" canaries

# Create "subsampled" train data => all non-canaries + in-canaries
subsampled_train_data = ConcatDataset([
    non_canaries,
    torch.utils.data.Subset(canaries, canaries_in_idx)
])

# DataLoaders
subsampled_train_loader = DataLoader(
    subsampled_train_data, 
    batch_size=4, 
    shuffle=True
)
canary_loader = DataLoader(
    canaries, 
    batch_size=4, 
    shuffle=False
)

In [15]:
def simple_train(model, device, train_loader, optimizer, criterion, epoch, client_id=None):
    model.train()
    loss_list = []
    for batch_idx, (data, target) in enumerate(train_loader):
        # Here, `data` is (input_ids, attention_mask) and `target` is the label
        # so we can destructure it:
        input_ids, attention_mask = data

        # Move to device
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        target = target.to(device)

        # Forward pass
        output = model(input_ids, attention_mask=attention_mask)
        # DistilBertForSequenceClassification => output.logits
        logits = output.logits

        # Loss
        loss = criterion(logits, target)

        # Backprop
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        loss_list.append(loss.item())

    print(f"Client {client_id} - Epoch {epoch}, Avg Loss: {np.mean(loss_list):.4f}")

In [ ]:
# Suppose we have a DistilBERT model, optimizer, etc.
import torch.nn as nn
from transformers import DistilBertForSequenceClassification, AdamW

model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
).to(device)

optimizer = AdamW(model.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()

# Train for some epochs
for epoch in range(1):
    simple_train(
        model=model, 
        device=device, 
        train_loader=subsampled_train_loader, 
        optimizer=optimizer, 
        criterion=criterion, 
        epoch=epoch, 
        client_id=1
    )